In [4]:
from pathlib import Path

OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W = 10
H = 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"

SURFACE_ALPHA = 0.28

GRID_STEP_U = 8
GRID_STEP_V = 8

GRID_GLOW_WIDTH = 1.4
GRID_CORE_WIDTH = 0.45

GRID_GLOW_ALPHA = 0.55
GRID_CORE_ALPHA = 0.95

COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

# Spherical Harmonic Surface

In [5]:
# Spherical Harmonic Rose — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/spherical_harmonic_rose_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "spherical_harmonic_rose_v1_grid"


# -----------------------------------------------------------------------------
# Spherical harmonic-like rose surface
#
# r(θ, φ) = 1 + A cos(kθ) sin(mφ)
#
# θ — azimuth angle
# φ — polar angle
# -----------------------------------------------------------------------------

theta = np.linspace(0.0, 2 * np.pi, 260)
phi = np.linspace(0.0, np.pi, 180)

TH, PH = np.meshgrid(theta, phi)

A = 0.38
K_THETA = 7
M_PHI = 4

R = 1.0 + A * np.cos(K_THETA * TH) * np.sin(M_PHI * PH)

X = R * np.sin(PH) * np.cos(TH)
Y = R * np.sin(PH) * np.sin(TH)
Z = R * np.cos(PH)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/spherical_harmonic_rose_v1_grid.webm
Frames: 192
Size: 4548.5 KB


[out#0/webm @ 0x14c805930] video:4546KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.044591%
frame=  192 fps= 26 q=32.0 Lsize=    4549KiB time=00:00:08.00 bitrate=4657.7kbits/s speed=1.09x    


# 3D Rose Surface

In [6]:
# 3D Rose Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/rose_surface_3d_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "rose_surface_3d_v1_grid"


# -----------------------------------------------------------------------------
# 3D rose surface
#
# r(θ, φ) = 1 + A1 cos(kθ) sin(mφ) + A2 sin(pθ) sin(qφ)
# -----------------------------------------------------------------------------

theta = np.linspace(0.0, 2 * np.pi, 280)
phi = np.linspace(0.0, np.pi, 190)

TH, PH = np.meshgrid(theta, phi)

A1 = 0.42
A2 = 0.18

K_THETA = 8
M_PHI = 5

P_THETA = 13
Q_PHI = 3

R = (
    1.0
    + A1 * np.cos(K_THETA * TH) * np.sin(M_PHI * PH)
    + A2 * np.sin(P_THETA * TH) * np.sin(Q_PHI * PH)
)

X = R * np.sin(PH) * np.cos(TH)
Y = R * np.sin(PH) * np.sin(TH)
Z = R * np.cos(PH)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    rn = (R - R.min()) / (R.max() - R.min() + 1e-9)

    brightness = 0.16 + 0.95 * rn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*R.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/rose_surface_3d_v1_grid.webm
Frames: 192
Size: 3778.7 KB


[out#0/webm @ 0x14af25bf0] video:3777KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.051404%
frame=  192 fps= 27 q=32.0 Lsize=    3779KiB time=00:00:08.00 bitrate=3869.4kbits/s speed=1.11x    


$$ r(\theta,\phi)=1+A_1\cos(k\theta)\sin(m\phi)+A_2\sin(p\theta)\sin(q\phi) $$

Это уже не классическая поверхность, а художественная angular-modulated форма: математический цветок / кристалл / радиальная гармоническая структура.

# Superquadric / Superellipsoid

In [7]:
# Superellipsoid — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/superellipsoid_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "superellipsoid_v1_grid"


# -----------------------------------------------------------------------------
# Superellipsoid geometry
#
# x = a * sgn(cos v)|cos v|^n1 * sgn(cos u)|cos u|^n2
# y = b * sgn(cos v)|cos v|^n1 * sgn(sin u)|sin u|^n2
# z = c * sgn(sin v)|sin v|^n1
# -----------------------------------------------------------------------------

u = np.linspace(-np.pi, np.pi, 240)
v = np.linspace(-np.pi / 2, np.pi / 2, 180)

U, V = np.meshgrid(u, v)

a = 1.0
b = 1.0
c = 1.15

N1 = 0.38
N2 = 0.38


def spow(x: np.ndarray, p: float) -> np.ndarray:
    return np.sign(x) * np.abs(x) ** p


X = a * spow(np.cos(V), N1) * spow(np.cos(U), N2)
Y = b * spow(np.cos(V), N1) * spow(np.sin(U), N2)
Z = c * spow(np.sin(V), N1)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/superellipsoid_v1_grid.webm
Frames: 192
Size: 4956.7 KB


[out#0/webm @ 0x144104230] video:4955KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.040997%
frame=  192 fps= 28 q=32.0 Lsize=    4957KiB time=00:00:08.00 bitrate=5075.6kbits/s speed=1.18x    


# Lissajous Knot / Lissajous Tube

In [8]:
# Lissajous Tube — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/lissajous_tube_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "lissajous_tube_v1_grid"


# -----------------------------------------------------------------------------
# Lissajous tube geometry
#
# Centerline:
# x = sin(a t + δ)
# y = sin(b t)
# z = sin(c t)
# -----------------------------------------------------------------------------

n_path = 520
n_tube = 48

t_path = np.linspace(0, 2 * np.pi, n_path)
theta = np.linspace(0, 2 * np.pi, n_tube)

A_FREQ = 3
B_FREQ = 4
C_FREQ = 5
DELTA = np.pi / 2

cx = np.sin(A_FREQ * t_path + DELTA)
cy = np.sin(B_FREQ * t_path)
cz = np.sin(C_FREQ * t_path)

C = np.vstack([cx, cy, cz]).T

dC = np.gradient(C, axis=0)
tangent = dC / np.linalg.norm(dC, axis=1, keepdims=True)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / np.linalg.norm(normal, axis=1, keepdims=True)

binormal = np.cross(tangent, normal)
binormal = binormal / np.linalg.norm(binormal, axis=1, keepdims=True)

tube_r = 0.075

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/lissajous_tube_v1_grid.webm
Frames: 192
Size: 4707.7 KB


[out#0/webm @ 0x12e624840] video:4706KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.042854%
frame=  192 fps= 31 q=32.0 Lsize=    4708KiB time=00:00:08.00 bitrate=4820.7kbits/s speed=1.28x    


Formula:

$$x=\sin(3t+\pi/2)$$
$$y=\sin(4t)$$
$$z=\sin(5t)$$

This creates a closed 3D Lissajous curve. The script thickens it into a tube, so it becomes a rotating orbital knot rather than just a line.

# Spherical Spirograph

In [9]:
# Spherical Spirograph — rotating mathematical sculpture
# Output: media-site/animations/Math/spherical_spirograph_v1.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "spherical_spirograph_v1"


# -----------------------------------------------------------------------------
# Spherical spirograph curve
# -----------------------------------------------------------------------------

n = 5000
t_curve = np.linspace(0, 2 * np.pi, n)

A = 7
B = 11
C = 5

theta = A * t_curve
phi = np.pi / 2 + 0.75 * np.sin(B * t_curve)

R = 1.0 + 0.16 * np.sin(C * t_curve)

X = R * np.sin(phi) * np.cos(theta)
Y = R * np.sin(phi) * np.sin(theta)
Z = R * np.cos(phi)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.7
Y = Y / scale * 3.7
Z = Z / scale * 3.7


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot(
        X, Y, Z,
        color=COL,
        linewidth=1.2,
        alpha=0.92,
    )

    ax.plot(
        X, Y, Z,
        color="#9ffcff",
        linewidth=4.0,
        alpha=0.10 + 0.08 * pulse,
    )

    # faint reference sphere
    u = np.linspace(0, 2 * np.pi, 80)
    v = np.linspace(0, np.pi, 40)
    U, V = np.meshgrid(u, v)

    SX = 3.15 * np.sin(V) * np.cos(U)
    SY = 3.15 * np.sin(V) * np.sin(U)
    SZ = 3.15 * np.cos(V)

    ax.plot_wireframe(
        SX, SY, SZ,
        rstride=8,
        cstride=8,
        color=COL,
        linewidth=0.25,
        alpha=0.08,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/spherical_spirograph_v1.webm
Frames: 192
Size: 1794.0 KB


[out#0/webm @ 0x158604ee0] video:1792KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.104188%
frame=  192 fps= 47 q=32.0 Lsize=    1794KiB time=00:00:08.00 bitrate=1837.1kbits/s speed=1.95x    


# Polar Flower Terrain

In [10]:
# Polar Flower Terrain — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/polar_flower_terrain_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "polar_flower_terrain_v1_grid"


# -----------------------------------------------------------------------------
# Polar flower terrain
#
# z = A * exp(-d r^2) * sin(k θ) * r^p
# -----------------------------------------------------------------------------

r = np.linspace(0.0, 2.2, 190)
theta = np.linspace(0.0, 2 * np.pi, 260)

R, T = np.meshgrid(r, theta)

K = 9
A = 1.0
DAMPING = 0.32
POWER = 1.6

X = R * np.cos(T)
Y = R * np.sin(T)
Z = A * np.exp(-DAMPING * R**2) * np.sin(K * T) * R**POWER

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-2.8, 2.8)

    ax.set_box_aspect((1, 1, 0.75))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=32 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/polar_flower_terrain_v1_grid.webm
Frames: 192
Size: 4817.2 KB


[out#0/webm @ 0x12fe06d10] video:4815KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.042185%
frame=  192 fps= 30 q=32.0 Lsize=    4817KiB time=00:00:08.00 bitrate=4932.8kbits/s speed=1.24x    
